In [1]:
import pandas as pd, numpy as np
from datetime import datetime
from pathlib import Path

DATA_DIR   = r"C:\Users\Prisha\Downloads\ml_turbofan\ml turbofan nasa\CMAPSSData"
OUT_DIR    = Path(r"C:\Users\Prisha\Downloads\ml_turbofan\data preprocessing") / "processed_output" / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
SUBSETS    = ["FD001", "FD002", "FD003", "FD004"]
TRAIN_FRAC = 0.70                      # share of engines used for training
A, B       = 60, 15                    # near-anomaly / anomaly RUL cutoffs
SEED       = 42

cols = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
sensors = [c for c in cols if c.startswith("s")]

frames = []
for s in SUBSETS:
    DATA_DIR = r"C:\Users\Prisha\Downloads\ml_turbofan\ml turbofan nasa\CMAPSSData"
    d = pd.read_csv(f"{DATA_DIR}/train_{s}.txt", sep=r"\s+", header=None, names=cols)
    d["subset"] = s
    d["unit"] = s + "_" + d["unit"].astype(str)          # unique engine id
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

# RUL and 3-class label
df["rul"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
df["label"] = np.select([df.rul <= B, df.rul <= A], ["anomaly", "near_anomaly"], "normal")

# normalize sensors per operating condition (matters for FD002/FD004)
df["cond"] = df[["op1", "op2", "op3"]].round(1).astype(str).agg("_".join, axis=1)
healthy = df[df.label == "normal"]
g = healthy.groupby(["subset", "cond"])[sensors]
mu, sd = g.transform("mean"), g.transform("std")
mu_map = healthy.groupby(["subset", "cond"])[sensors].mean()
sd_map = healthy.groupby(["subset", "cond"])[sensors].std().replace(0, 1)
idx = pd.MultiIndex.from_frame(df[["subset", "cond"]])
df[sensors] = (df[sensors].values - mu_map.reindex(idx).values) / sd_map.reindex(idx).values
df = df.dropna(subset=sensors)

# split by engine
units = df["unit"].unique()
rng = np.random.default_rng(SEED)
train_units = set(rng.choice(units, int(len(units) * TRAIN_FRAC), replace=False))
df["split"] = np.where(df["unit"].isin(train_units), "train", "test")

OUT_DIR.mkdir(parents=True, exist_ok=False)
df.to_csv(OUT_DIR / "turbofan_full.csv", index=False)
# CANARI is fit on healthy rows from training engines only
df[(df.split == "train") & (df.label == "normal")].to_csv(OUT_DIR / "train_normal.csv", index=False)
df[df.split == "test"].to_csv(OUT_DIR / "test_all.csv", index=False)
print("Saved outputs to:", OUT_DIR)
print(df.groupby(["split", "label"]).size())

Saved outputs to: C:\Users\Prisha\Downloads\ml_turbofan\data preprocessing\processed_output\run_20261103_183930_701358
split  label       
test   anomaly          3408
       near_anomaly     9585
       normal          35134
train  anomaly          7936
       near_anomaly    22320
       normal          81976
dtype: int64


In [2]:
df.head()

,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s17,s18,s19,s20,s21,subset,rul,label,cond,split
0,FD001_1,1,-0.0007,-0.0004,100.0,0.0,-1.732642,0.327432,-0.680326,0.000000e+00,...,-0.512520,0.0,0.0,1.254049,1.043952,FD001,191,normal,-0.0_-0.0_100.0,test
1,FD001_1,2,0.0019,-0.0003,100.0,0.0,-0.880101,0.774517,-0.284214,0.000000e+00,...,-0.484038,0.0,0.0,0.793973,1.101880,FD001,190,normal,0.0_-0.0_100.0,test
2,FD001_1,3,-0.0043,0.0003,100.0,0.0,-0.357625,-0.062852,-0.125358,0.000000e+00,...,-2.192144,0.0,0.0,0.459096,0.128833,FD001,189,normal,-0.0_0.0_100.0,test
3,FD001_1,4,0.0007,0.0000,100.0,0.0,-0.350710,-1.136612,-0.534163,1.776357e-15,...,-0.523591,0.0,0.0,-0.079246,0.506424,FD001,188,normal,0.0_0.0_100.0,test
4,FD001_1,5,-0.0019,-0.0002,100.0,0.0,-0.277000,-1.096491,0.214145,0.000000e+00,...,0.347533,0.0,0.0,0.070009,0.863906,FD001,187,normal,-0.0_-0.0_100.0,test
